# M4.2 Solenoids and Ampere's Law on a Grid

Book pointer: chapter 12, sections 12.1 (magnetic flux), 12.2 (Ampere's law), 12.4 (proof).

## 1. Recall

1. State Ampere's law in integral form and in differential form. Which theorem connects them?
2. Why is $\nabla\cdot\mathbf{B} = 0$, and what does it say about the flux of $\mathbf{B}$ through any closed surface?
3. Draw the Amperian loop for an infinite solenoid with $n$ turns per length and current $I$. Which sides contribute, and what is $B$ inside?
4. The right-hand side of Ampere's law is "the current through the surface spanning the loop". For a loop that threads a solenoid, which windings count, and why does the choice of spanning surface not matter?
5. A closed polyline has vertices $\mathbf{r}_0, \dots, \mathbf{r}_{N-1}$. Write the midpoint-rule estimate of $\oint\mathbf{B}\cdot d\mathbf{l}$ as a sum.

## 2. Principle to algorithm

A solenoid is a wire wound into a helix. `biot_savart.helix` returns its vertices, `biot_savart.segments` turns them into midpoints and step vectors $\Delta\mathbf{l}$, and `biot_savart.wire_B` sums the Biot-Savart contributions $\mu_0 I\,\Delta\mathbf{l}\times\mathbf{R}/4\pi|\mathbf{R}|^3$ over every segment for every observation point. That is the whole field computation; there is no separate solenoid formula in the code. The closed forms `solenoid_axis_B` and `straight_wire_B` exist only to test it.

Ampere's law, $\oint\mathbf{B}\cdot d\mathbf{l} = \mu_0 I_{\text{enc}}$, is checked the way it is stated: a closed polyline is sampled, the field is evaluated at the midpoints of its steps, and `fields.line_integral` adds up $\mathbf{B}\cdot\Delta\mathbf{l}$. The right side counts the windings that cross any surface spanning the loop. The loop must not pass close to a wire relative to the segment length used for that wire, so where a loop threads the windings the helix is refined locally and the loop is sampled finely.

$\nabla\cdot\mathbf{B} = 0$ is checked with the same tools as Gauss' law in module 2: `fields.divergence` of the gridded field, and `fields.box_flux` through a box inside the solenoid, which must vanish because no field lines start or end anywhere.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields
from emlib import biot_savart as bs
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# A finite solenoid: radius 5 cm, length 50 cm, 200 turns, 1 A.
R, L, turns, I = 0.05, 0.5, 200, 1.0
n_density = turns / L                      # turns per metre
pitch = L / turns                          # axial advance per turn
helix_pts = bs.helix(R, L, turns, n_per_turn=36)
mid, dl = bs.segments(helix_pts, closed=False)
B_sol = lambda pts: bs.wire_B(mid, dl, I, pts)
print(f"{len(mid)} straight segments, pitch {pitch * 1e3:.1f} mm, segment length {np.linalg.norm(dl, axis=1).mean() * 1e3:.1f} mm")
print(f"infinite-solenoid field mu0 n I = {MU0 * n_density * I * 1e3:.4f} mT")

### The field on the axis

For an infinite solenoid Ampere's law gives $B = \mu_0 n I$ inside and zero outside. A finite one falls short of that at the centre and drops to about half of it at each end, because an end sees only the windings on one side. The closed form `solenoid_axis_B` is the on-axis field of a uniform surface current; the helix differs from it by the discreteness of the turns and by the straight-segment approximation of each turn. The second effect dominates here: with 36 segments per turn the centre field comes out a quarter of a percent high, and the excess falls by a factor of four each time the segments are halved, which is the signature of a second-order discretisation error.

In [ ]:
z = np.linspace(-0.4, 0.4, 161)
axis_pts = np.stack([0 * z, 0 * z, z], axis=-1)
B_axis = B_sol(axis_pts)
B_exact = bs.solenoid_axis_B(R, L, n_density, I, z)
B_inf = MU0 * n_density * I

i_c, i_end = np.argmin(np.abs(z)), np.argmin(np.abs(z - L / 2))
print(f"centre: B_z / (mu0 n I) = {B_axis[i_c, 2] / B_inf:.4f}   closed form {B_exact[i_c] / B_inf:.4f}")
print(f"end:    B_z / (mu0 n I) = {B_axis[i_end, 2] / B_inf:.4f}   closed form {B_exact[i_end] / B_inf:.4f}")
print(f"max relative deviation of the helix from the closed form on the axis: {np.abs(B_axis[:, 2] / B_exact - 1).max():.2e}")
print(f"largest transverse component on the axis / (mu0 n I): {np.abs(B_axis[:, :2]).max() / B_inf:.1e}   (a helix is not exactly axisymmetric)")
for n_per_turn in (36, 72, 144):
    m_t, d_t = bs.segments(bs.helix(R, L, turns, n_per_turn=n_per_turn), closed=False)
    excess = bs.wire_B(m_t, d_t, I, np.array([[0.0, 0.0, 0.0]]))[0, 2] / bs.solenoid_axis_B(R, L, n_density, I, 0.0) - 1
    print(f"  {n_per_turn:3d} segments per turn: centre field exceeds the closed form by {excess:.2e}")

plt.plot(z, B_axis[:, 2] / B_inf, "C0", lw=2, label="Biot-Savart sum over the helix")
plt.plot(z, B_exact / B_inf, "k--", label="closed form, uniform surface current")
plt.axhline(1, color="grey", ls=":", label="mu0 n I")
plt.axvline(-L / 2, color="grey", lw=0.5); plt.axvline(L / 2, color="grey", lw=0.5)
plt.xlabel("z  [m]"); plt.ylabel("B_z / (mu0 n I)"); plt.legend(); plt.title("on the axis of a finite solenoid"); plt.show()

### The field in a cross-section

The field in the plane $y = 0$ containing the axis. Inside, the field lines run parallel to the axis; at the ends they spread out and close through the outside, where the field is weak but not zero. The windings cross this plane at $x = \pm R$, so $|\mathbf{B}|$ spikes there and the colour scale is logarithmic.

In [ ]:
xs = np.linspace(-0.12, 0.12, 31)
zs = np.linspace(-0.4, 0.4, 61)
X, Z = np.meshgrid(xs, zs, indexing="ij")
plane = np.stack([X, np.zeros_like(X), Z], axis=-1)
B_plane = B_sol(plane)
Bmag = np.linalg.norm(B_plane, axis=-1)

fig, ax = plt.subplots(figsize=(5, 9))
im = ax.pcolormesh(X, Z, np.log10(Bmag / B_inf), cmap="magma", vmin=-3, vmax=0.5, shading="auto")
ax.streamplot(xs, zs, B_plane[..., 0].T, B_plane[..., 2].T, color="w", linewidth=0.6, density=1.1)
ax.plot([R, R, -R, -R], [-L / 2, L / 2, -L / 2, L / 2], "c|", ms=8)
ax.set_aspect("equal"); ax.set_xlabel("x  [m]"); ax.set_ylabel("z  [m]"); ax.set_title("log10 |B| / (mu0 n I) in the plane y = 0")
fig.colorbar(im, ax=ax, shrink=0.6); plt.show()
print(f"|B| at (x, z) = (0.10, 0), outside the windings: {Bmag[np.argmin(np.abs(xs - 0.10)), np.argmin(np.abs(zs))] / B_inf:.2e} of mu0 n I")

### Circulation around loops that thread the windings

The Amperian rectangle for a solenoid lies in the plane $y = 0$: one side along the axis, one side outside at $x = 0.15\,$m, and two short sides that cross the windings at $x = R$. With the loop oriented so that its normal is $+\hat{\mathbf{y}}$, the enclosed windings are those crossing the half-plane $x > 0$ between the two short sides, each carrying $I$ in the $+y$ direction, so $\oint\mathbf{B}\cdot d\mathbf{l}$ should be $\mu_0 I$ times the number of those crossings.

Two numerical precautions. The short sides must pass between turns, so their height is set to a half-integer number of pitches, which puts the wire $1.25\,$mm away. At that distance the $8.7\,$mm straight segments of the coarse helix are far too crude, so the two turns nearest each crossing are re-sampled with 1500 segments per turn, and the loop itself is sampled every $0.02\,$mm within $5\,$mm of the crossing. Everywhere else both can be coarse.

In [ ]:
def refined_helix(crossing_heights, fine=1500, coarse=36):
    '''Segments of the same helix, with the two turns around each crossing height sampled fine.'''
    t = np.linspace(0, 2 * np.pi * turns, turns * coarse + 1)
    for zc in crossing_heights:
        k = (zc + L / 2) / pitch                                 # turn number at height zc
        t = np.concatenate([t, np.linspace(2 * np.pi * (k - 1), 2 * np.pi * (k + 1), 2 * fine + 1)])
    t = np.unique(t); t = t[(t >= 0) & (t <= 2 * np.pi * turns)]
    pts = np.stack([R * np.cos(t), R * np.sin(t), -L / 2 + L * t / (2 * np.pi * turns)], axis=-1)   # same curve as bs.helix
    return bs.segments(pts, closed=False)

def rectangle_xz(x0, x1, z0, z1, x_cross=None):
    '''Vertices of a rectangle in the plane y = 0, traversed with normal +y: up the x0 side, across at z1, down the x1 side, back at z0.'''
    if x_cross is None:
        xs = np.linspace(x0, x1, 200, endpoint=False)
    else:
        xs = np.concatenate([np.linspace(x0, x_cross - 0.005, 60, endpoint=False),
                             np.linspace(x_cross - 0.005, x_cross + 0.005, 500, endpoint=False),
                             np.linspace(x_cross + 0.005, x1, 60, endpoint=False)])
    zs = np.linspace(z0, z1, 200, endpoint=False)
    side1 = np.stack([np.full_like(zs, x0), 0 * zs, zs], axis=-1)
    side2 = np.stack([xs, 0 * xs, np.full_like(xs, z1)], axis=-1)
    side3 = np.stack([np.full_like(zs, x1), 0 * zs, zs[::-1]], axis=-1)
    side4 = np.stack([xs[::-1], 0 * xs, np.full_like(xs, z0)], axis=-1)
    return np.vstack([side1, side2, side3, side4])

def enclosed_turns(z1):
    '''Windings crossing the half-plane x > 0, y = 0 with |z| < z1.'''
    return int(np.sum(np.abs(-L / 2 + np.arange(turns + 1) * pitch) < z1))

results = {}
for z1 in (0.10125, 0.05125):                                    # half-integer pitches: 40.5 and 20.5 pitches from the centre
    m_f, d_f = refined_helix([z1, -z1])
    loop_pts = rectangle_xz(0.0, 0.15, -z1, z1, x_cross=R)
    closed = np.vstack([loop_pts, loop_pts[:1]])
    circ = fields.line_integral(lambda p: bs.wire_B(m_f, d_f, I, p), closed)
    results[z1] = (circ, enclosed_turns(z1))
    print(f"half-height {z1:.5f} m: circulation / (mu0 I) = {circ / (MU0 * I):8.3f}   windings enclosed = {enclosed_turns(z1)}   ({len(m_f)} segments, {len(loop_pts)} loop points)")

Both loops come out a quarter of a percent above the count of enclosed windings, the same excess as on the axis and for the same reason: the coarse turns are 36-gons, and the straight-segment sum overestimates the field of a circle at second order in the segment length. Halving the coarse segments cuts the excess to $0.06$ percent. The sides that cross the windings contribute almost nothing on their own (section 4 shows the breakdown), so the result does not depend on how carefully the crossing is placed once the local refinement is in place.

### Exercise: circulation of a field around a closed polyline

Replace the body of `ampere_circulation` so that it returns $\oint\mathbf{B}\cdot d\mathbf{l}$ for a callable field around the polyline `path` of shape (N, 3), treating the polyline as closed (the last vertex joins the first). The check uses a circle around a long straight wire, where the answer is $\mu_0 I$.

In [ ]:
def ampere_circulation(B_func, path):
    '''Line integral of B_func around the closed polyline path (N, 3).'''
    closed = np.vstack([path, path[:1]])                         # replace from here with your own version
    return fields.line_integral(B_func, closed)


L_wire = 400.0
m_w, d_w = bs.segments(bs.straight_wire([0, 0, -L_wire], [0, 0, L_wire], 20000), closed=False)
B_wire = lambda p: bs.wire_B(m_w, d_w, I, p)
t = np.linspace(0, 2 * np.pi, 2000, endpoint=False)
circle = np.stack([0.7 * np.cos(t), 0.7 * np.sin(t), 0 * t], axis=-1)
c = ampere_circulation(B_wire, circle)
print(f"circulation around the wire / (mu0 I) = {c / (MU0 * I):.6f}")
assert abs(c / (MU0 * I) - 1) < 1e-3
not_enclosing = np.stack([3.0 + 0.5 * np.cos(t), 0.5 * np.sin(t), 0 * t], axis=-1)
print(f"circulation around a loop beside the wire / (mu0 I) = {ampere_circulation(B_wire, not_enclosing) / (MU0 * I):.1e}")

A loop outside the solenoid that threads nothing. The rectangle between $x = 0.08$ and $x = 0.15$ sits in the weak external field. Its circulation is not zero because the field vanishes but because the contributions along the two long sides cancel against the short ones, as they must for any loop enclosing no current.

In [ ]:
outside = rectangle_xz(0.08, 0.15, -0.1, 0.1)
c_out = ampere_circulation(B_sol, outside)
print(f"loop enclosing no windings: circulation / (mu0 I) = {c_out / (MU0 * I):.1e}")
print(f"for scale, the loop through the windings of the same height gave {results[0.10125][0] / (MU0 * I):.1f}")

## 4. Verify

Three closed-form checks with asserts, then the circulation integrand along the threading loop.

1. A long straight wire against $\mu_0 I/2\pi r$ at three radii.
2. $\nabla\cdot\mathbf{B} = 0$: the net flux of the solenoid field through an off-centre box inside the solenoid, relative to the flux through one of its end faces, and the gridded divergence relative to $|\mathbf{B}|/h$.
3. The two threading loops: circulation equals the number of enclosed windings times $\mu_0 I$ within one percent.

In [ ]:
# 1. Straight wire. Segments of 1 cm, so the sampling radii are at least 20 segment lengths from the wire.
m_l, d_l = bs.segments(bs.straight_wire([0, 0, -500.0], [0, 0, 500.0], 100000), closed=False)
r = np.array([0.2, 1.0, 3.0])
B_r = bs.wire_B(m_l, d_l, I, np.stack([r, 0 * r, 0 * r], axis=-1))
rel = np.abs(B_r[:, 1] / bs.straight_wire_B(I, r) - 1)
print("straight wire: relative error at r =", r, ":", np.array2string(rel, formatter={"float": lambda v: f"{v:.1e}"}))
assert rel.max() < 1e-3

# 2. No magnetic monopoles: box flux and divergence inside the solenoid.
# The box is off centre on purpose: a box centred on the axis would give zero by symmetry alone.
x3, y3, z3, P3 = grids.grid3d((-0.01, 0.035), (-0.02, 0.025), (-0.12, 0.08), (11, 11, 21))
h3 = grids.spacing(x3, y3, z3)
B3 = B_sol(P3)
flux = fields.box_flux(B3, h3, (0, 0, 0), (10, 10, 20))
end_face = np.trapezoid(np.trapezoid(B3[:, :, -1, 2], dx=h3[1], axis=1), dx=h3[0])    # flux through the top face alone
Bmean = np.linalg.norm(B3, axis=-1).mean()
div3 = fields.divergence(B3, *h3)[1:-1, 1:-1, 1:-1]
print(f"net box flux / flux through one end face = {flux / abs(end_face):.1e}     max |div B| h / |B| = {np.abs(div3).max() * h3[0] / Bmean:.1e}")
assert abs(flux) < 1e-3 * abs(end_face)

# 3. Ampere's law for the threading loops.
for z1, (circ, k) in results.items():
    print(f"half-height {z1:.5f}: circulation / (k mu0 I) = {circ / (k * MU0 * I):.4f}   with k = {k}")
    assert abs(circ / (k * MU0 * I) - 1) < 1e-2

In [ ]:
# Where the circulation comes from: B . dl step by step around the loop of half-height 0.10125.
z1 = 0.10125
m_f, d_f = refined_helix([z1, -z1])
loop_pts = rectangle_xz(0.0, 0.15, -z1, z1, x_cross=R)
closed = np.vstack([loop_pts, loop_pts[:1]])
steps = np.diff(closed, axis=0); mids = 0.5 * (closed[1:] + closed[:-1])
contrib = np.einsum("nk,nk->n", bs.wire_B(m_f, d_f, I, mids), steps)
s = np.concatenate([[0], np.cumsum(np.linalg.norm(steps, axis=1))])[:-1]
cum = np.cumsum(contrib) / (MU0 * I)
n_side = 200
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s, cum, "C0")
for name, pos in (("axis", 0), ("top side", s[n_side]), ("outer side", s[n_side + 620]), ("bottom side", s[2 * n_side + 620])):
    ax.axvline(pos, color="grey", lw=0.5); ax.text(pos + 0.005, 5, name, fontsize=8)
ax.set_xlabel("distance along the loop  [m]"); ax.set_ylabel("cumulative B . dl / (mu0 I)")
ax.set_title("the axis side supplies almost all of the circulation"); plt.show()
print(f"share of the circulation from the axis side: {contrib[:n_side].sum() / contrib.sum():.3f}")
print(f"share from the two sides that cross the windings: {(contrib[n_side:n_side + 620].sum() + contrib[2 * n_side + 620:].sum()) / contrib.sum():.3f}")

## 5. Explore and rebuild

Explore. Keep $n$ and $I$ fixed and vary the aspect ratio $L/R$ from 2 to 40. For each solenoid, measure the field at the centre and at a quarter of the way to the end, both divided by $\mu_0 n I$, and plot them against $L/R$. Find the aspect ratio at which the centre reaches 99 percent of the infinite-solenoid value. Then bend a 200-turn helix into a torus of major radius $0.3\,$m and sample $|\mathbf{B}|$ along a radial line through the tube: inside the tube it should fall as $1/r$, which is what Ampere's law predicts for a circular Amperian loop threading all $N$ turns, and outside it should be close to zero. Decide how close, and why it is not exactly zero.

Rebuild. Close this notebook and, from a blank editor, write a helix builder, the circulation function for a closed polyline, and the comparison of the end field to half of the centre field.
